## imports:

In [3]:
import json
import os
from pathlib import Path

import duckdb



## audit:


In [4]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW, PROC, REP = ROOT/"data/raw", ROOT/"data/processed", ROOT/"reports"
for d in (RAW, PROC, REP):
    d.mkdir(parents=True, exist_ok=True)

SPLIT_SEED = 2026
MASK_SEED  = "digikala-semi-supervised-v1"
MASK_RATE  = 0.20

CSV = RAW / "digikala-comments.csv"
print(CSV, "exists:", CSV.exists(), "| size GB:", round(CSV.stat().st_size/1e9, 2))

e:\maktab seesion\home work\code\project-3\data\raw\digikala-comments.csv exists: True | size GB: 1.28


In [6]:
con = duckdb.connect()
con.execute("SET memory_limit='6GB';")            
con.execute(f"SET threads={os.cpu_count()};")
con.execute(f"SET temp_directory='{(PROC/'duck_tmp').as_posix()}';")

PQ = PROC / "comments.parquet"
con.execute(f"""
COPY (
  SELECT * FROM read_csv('{CSV.as_posix()}', header=true, all_varchar=true,quote='"', escape='"')) TO '{PQ.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 200000)""")
P = f"read_parquet('{PQ.as_posix()}')"

print(con.execute(f"DESCRIBE SELECT * FROM {P}").df()[["column_name","column_type"]])
print(con.execute(f"SELECT count() AS n_rows FROM {P}").df())
print(con.execute(f"SELECT * FROM {P} LIMIT 3").df().T)

              column_name column_type
0                      id     VARCHAR
1                   title     VARCHAR
2                    body     VARCHAR
3              created_at     VARCHAR
4                    rate     VARCHAR
5   recommendation_status     VARCHAR
6                is_buyer     VARCHAR
7              product_id     VARCHAR
8              advantages     VARCHAR
9           disadvantages     VARCHAR
10                  likes     VARCHAR
11               dislikes     VARCHAR
12           seller_title     VARCHAR
13            seller_code     VARCHAR
14      true_to_size_rate     VARCHAR
    n_rows
0  6156289
                                     0  \
id                            53672599   
title                   پیشنهاد نمیشود   
body                   به درد نمیخوره    
created_at              23 شهریور 1402   
rate                               1.0   
recommendation_status  not_recommended   
is_buyer                          True   
product_id                      25

In [ ]:
# con = duckdb.connect()
# p = f"read_paraquet('{(PROC/'comments.paraquet').as_posix()}')"

In [10]:
expected = ["id","title","body","created_at","rate","recommendation_status",
            "is_buyer","product_id","advantages","disadvantages","likes",
            "dislikes","seller_title","seller_code","true_to_size_rate"]
actual = con.execute(f"DESCRIBE SELECT * FROM {P}").df()["column_name"].tolist()
print("missing columns:", [c for c in expected if c not in actual])
print("extra columns  :", [c for c in actual if c not in expected])

gate = con.execute(f"""
SELECT
  count(*)                                                       AS n_rows,
  count(*) - count(DISTINCT id)                                  AS dup_ids,
  count(*) FILTER (WHERE body IS NULL OR trim(body) = '')        AS empty_body,
  count(*) FILTER (WHERE rate IS NULL OR trim(rate) = '')        AS rate_missing,
  count(*) FILTER (WHERE trim(rate) <> '' AND TRY_CAST(rate AS DOUBLE) IS NULL) AS rate_non_numeric,
  count(*) FILTER (WHERE TRY_CAST(rate AS DOUBLE) NOT BETWEEN 0 AND 5)          AS rate_out_of_range,
  min(TRY_CAST(rate AS DOUBLE))                                  AS rate_min,
  max(TRY_CAST(rate AS DOUBLE))                                  AS rate_max,
  count(DISTINCT rate)                                           AS rate_n_unique,
  count(DISTINCT product_id)                                     AS n_products,
  count(DISTINCT seller_code)                                    AS n_sellers
FROM {P}
""").df().T
print(gate)

labels = con.execute(f"""
SELECT recommendation_status, count(*) AS n FROM {P}
GROUP BY 1 ORDER BY n DESC""").df()
print(labels)

rates = con.execute(f"""
SELECT 
    rate,
    count(*) AS n,
    round(count(*) * 100.0 / sum(count(*)) OVER (), 2) AS pct,
    round(sum(count(*)) OVER (ORDER BY count(*) DESC) * 100.0 / sum(count(*)) OVER (), 2) AS cumulative_pct
FROM {P}
GROUP BY rate
ORDER BY n DESC
LIMIT 20
""").df()
print(rates)

gate.to_csv(REP/"quality_gate.csv")
labels.to_csv(REP/"label_vocab.csv", index=False)
rates.to_csv(REP/"rate_distribution.csv", index=False)

missing columns: []
extra columns  : []
                           0
n_rows             6156289.0
dup_ids               3229.0
empty_body               0.0
rate_missing             0.0
rate_non_numeric         0.0
rate_out_of_range        1.0
rate_min                 0.0
rate_max              2500.0
rate_n_unique          178.0
n_products          331599.0
n_sellers            29214.0
  recommendation_status        n
0           recommended  4162839
1                   nan   894426
2               no_idea   594121
3       not_recommended   504903
    rate        n    pct  cumulative_pct
0    5.0  2398801  38.97           38.97
1    4.0  1515723  24.62           63.59
2    3.0   958799  15.57           79.16
3    0.0   529098   8.59           87.75
4    1.0   327662   5.32           93.08
5    2.0   198077   3.22           96.29
6    4.5    13479   0.22           96.51
7    3.5    12207   0.20           96.71
8   4.25    11653   0.19           96.90
9   3.75    10045   0.16           97